# Water availability (f) — full pipeline notebook
Settings, a quick check with plots on a small area, then the periodic maps for whole areas (national, regional, provincial or municipal).

## Method
A daily root-zone water balance on a harmonized **1 ha grid** (no model to train):

`GPM IMERG V07 P + ERA5-Land weather (FAO-56 ET0) + ESA WorldCover cropland + OpenLandMap soil (Earth Engine, computePixels) → harmonize grid & units → SCS-CN runoff, drainage, ETc = Ks·Kc·ET0 → S → f`

```
P + I − R − ET − D = ΔS          f = (S − WP) / (FC − WP)
```

* **Stage 1 — harmonize.** Every layer is requested straight on the 1 ha grid and converted to water-balance units (mm, °C, kPa, MJ m⁻²). WorldCover's 10 m cropland is turned into the cropland share of each 1 ha cell. Tiles are cached, so a rerun downloads nothing again.
* **Stage 2 — water balance.** Daily per pixel. The soil storage at the end of each period is saved, so the next 14-day run only needs those 14 days.
* **Tiling.** The area is cut into `TILE_PX` tiles, and each tile's daily stack is requested in chunks that stay under the computePixels request limit. A chunk that's still too large is split in half and retried.

Output per area and period (`CADENCE`: 14-day windows or semimonthly): `wa_{AREA}_{PERIOD}_status.tif`, `_balance.tif` (f, P, I, R, ET, D, ΔS …), `_summary.csv` (hectares per class), `_stats.csv`, `_run.json`. After a national or regional run, the areas are merged into one mosaic per period.

Status classes (f at the end of the period): −1 no data · 1 severe deficit (f < 0.25) · 2 moderate deficit (0.25 ≤ f < 1 − p) · 3 adequate (1 − p ≤ f ≤ 1) · 4 surplus (f > 1).

In [ ]:
# the repository (folder that contains waterbalance/):
#  - a copy on Drive: set REPO_DIR to it, or
#  - clone it here: private repo -> add a read-only GitHub token as the Colab secret GITHUB_TOKEN
import os, subprocess
REPO_DIR = '/content/ai-waterbalance'
REPO_URL = 'github.com/Leads-DigiSaka-System/ai-waterbalance.git'
BRANCH   = 'claude/modest-noether-jwyte4'   # change to 'main' once merged
if not os.path.isdir(os.path.join(REPO_DIR, 'waterbalance')):
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://x-access-token:{tok}@{REPO_URL}', REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)  # don't keep the token
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
!pip install -q -r {REPO_DIR}/requirements.txt

## Import Libraries

In [ ]:
import os, sys, json, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=RuntimeWarning)
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
sys.path.insert(0, REPO_DIR)
from waterbalance import RunConfig, Runner, CROPS, DATASETS
from waterbalance import balance as bal, periods as per

# ---- Earth Engine ----
EE_PROJECT = 'sensesaka2024'      # Google Cloud project registered for Earth Engine

# ---- area ----
# 'national'   : every feature of VECTOR_PATH (one map per NAME_COL value), then one mosaic
# 'regional'   : the features whose REGION_COL is in AREA_NAMES, then one mosaic
# 'provincial' : the features whose NAME_COL is in AREA_NAMES (provinces file)
# 'municipal'  : same, with a municipal boundary file
# 'bbox'       : one rectangle, BBOX
AREA_LEVEL = 'provincial'
AREA_NAMES = ['NUEVA ECIJA']
EXCLUDE    = []                  # areas to skip (name contains any of these), e.g. ['PALAWAN']
VECTOR_PATH = '/content/drive/MyDrive/AGRI/Crop_Growth/vector/boundary_province_philippines.gpkg'
NAME_COL, REGION_COL = 'Pro_Name', 'Reg_Name'
# municipal example:
# VECTOR_PATH = '/content/drive/MyDrive/AGRI/Crop_Growth/vector/boundary_municipal_philippines.gpkg'
# NAME_COL = 'Mun_Name'; AREA_LEVEL = 'municipal'; AREA_NAMES = ['SCIENCE CITY OF MUÑOZ']
BBOX = (120.85, 15.55, 121.05, 15.75)   # used when AREA_LEVEL = 'bbox'

# ---- grid (harmonized resolution) ----
CRS          = 'EPSG:4326'      # global grid like ai-cropgrowth (~0.97 ha at 15 N); 'EPSG:32651' = exactly 1 ha
RESOLUTION_M = 100              # 100 m = 1 ha
TILE_PX      = 128              # tile side in pixels (128 px = 12.8 km); lower it if requests are too large
TILE_WORKERS = 2                # tiles processed in parallel
CROP_FRACTION_MIN = 0.5         # a 1 ha cell is cropland when >= 50 % of its WorldCover 10 m pixels are class 40

# ---- crop / model ----
CROP       = 'rice'             # 'generic' | 'rice' | 'maize'  (Kc, root depth Zr, depletion p: CROPS)
KC = ZR = P_DEPLETION = None    # overrides of the crop preset (None = preset)
IRRIGATION = 'none'             # 'none' (rainfed) | 'auto' (refill to field capacity when f < 1 - p: a scenario)
SEVERE_F   = 0.25               # f below this = severe deficit

# ---- periods ----
CADENCE       = '14d'           # '14d' (14-day windows from PERIOD_ANCHOR) | 'semimonthly' (1st-15th, 16th-end)
PERIOD_ANCHOR = '2026-01-01'
AS_OF         = None            # None = today; the default period is the last one that has ended by AS_OF
PERIODS       = None            # None = last complete period; or labels, e.g. ['20261008', '20261022'] (backfill in order)
SPINUP_DAYS   = 365             # spin-up when a tile has no saved soil state yet
PRECIP_SOURCE   = 'gpm'         # 'gpm' (IMERG V07: half-hourly -> daily, ~1 day latency; Late run flagged provisional)
                                #  | 'chirps' (CHIRPS daily, final only: weeks behind)
PRECIP_FALLBACK = 'era5'        # days the source has no data yet: use ERA5-Land precipitation (flagged provisional)

# ---- outputs ----
OUTPUT_TARGET = 'gdrive'        # 'gcs' (Cloud Storage) | 'gdrive' (Google Drive) | 'both' | 'local'
GCS_BUCKET = 'leads-agri'
GCS_PREFIX = 'products/water_availability'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/AI-WaterBalance/outputs'   # 'gdrive' / 'both'
LOCAL_OUTPUT_DIR  = '/content/waterbalance_outputs'                    # 'local'; staging for 'gcs'
WORK_DIR = '/content/drive/MyDrive/AI-WaterBalance/work'    # harmonized tiles + soil state (keep on Drive between runs)
MOSAIC_AFTER = True             # national / regional: one mosaic per period
MOSAIC_LABEL = None             # None = 'PHILIPPINES' (national) or the region name(s)

AS_OF_DATE = pd.Timestamp(AS_OF or pd.Timestamp.today()).normalize()
print(f'{AREA_LEVEL} {AREA_NAMES if AREA_LEVEL != "bbox" else BBOX} | {RESOLUTION_M:g} m grid ({CRS}) | '
      f'tile {TILE_PX} px | crop {CROP} {CROPS[CROP]} | output -> {OUTPUT_TARGET}')

In [ ]:
import ee
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('✓ Earth Engine ready')

In [ ]:
NEED_GCS = OUTPUT_TARGET in ('gcs', 'both')   # Cloud Storage sign-in only when saving there
if NEED_GCS:
    !gcloud auth application-default login manager@leadsagri.app

In [ ]:
if NEED_GCS:
    !gcloud auth list

In [ ]:
if NEED_GCS:
    import google.auth
    import google.auth.transport.requests

    from google.cloud import storage
    from google.auth import impersonated_credentials

    source_credentials, _ = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )

    target_credentials = impersonated_credentials.Credentials(
        source_credentials=source_credentials,
        target_principal="leadsagri-ai@sensesaka2024.iam.gserviceaccount.com",
        target_scopes=[
            "https://www.googleapis.com/auth/devstorage.read_write"
        ],
        lifetime=3600,
    )

    target_credentials.refresh(
        google.auth.transport.requests.Request()
    )

    print("✓ Impersonation successful")

In [ ]:
gcs_client = None
if NEED_GCS:
    gcs_client = storage.Client(project="sensesaka2024", credentials=target_credentials)
    print("✓ GCS client created")

## Set up the runner (settings + output destination)

In [ ]:
cfg = RunConfig(
    bbox=list(BBOX) if AREA_LEVEL == 'bbox' else None,
    aoi_path=None if AREA_LEVEL == 'bbox' else VECTOR_PATH,
    aoi_name_col=NAME_COL, region_col=REGION_COL,
    crs=CRS, resolution_m=RESOLUTION_M, tile_px=TILE_PX, tile_workers=TILE_WORKERS,
    ee_project=EE_PROJECT, crop_fraction_min=CROP_FRACTION_MIN,
    crop=CROP, kc=KC, zr=ZR, p=P_DEPLETION, irrigation=IRRIGATION, severe_f=SEVERE_F,
    cadence=CADENCE, period_anchor=PERIOD_ANCHOR, spinup_days=SPINUP_DAYS,
    precip_source=PRECIP_SOURCE, precip_fallback=PRECIP_FALLBACK,
    output_target=OUTPUT_TARGET, gcs_bucket=GCS_BUCKET, gcs_prefix=GCS_PREFIX,
    drive_root=DRIVE_OUTPUT_ROOT, output_dir=LOCAL_OUTPUT_DIR, work_dir=WORK_DIR,
    mosaic_after=MOSAIC_AFTER, mosaic_label=MOSAIC_LABEL,
)
runner = Runner(cfg, gcs_client=gcs_client)
RUN_PERIODS = runner.resolve_periods(PERIODS, AS_OF_DATE.date().isoformat())
print('outputs ->', cfg.output_root(), f'(+ gs://{GCS_BUCKET}/{GCS_PREFIX})' if NEED_GCS else '')
print('periods ->', [f'{p.label} ({p.start.date()} .. {(p.end - pd.Timedelta(days=1)).date()})' for p in RUN_PERIODS])
if AREA_LEVEL != 'bbox':
    print(f'{len(runner.list_areas())} areas in {VECTOR_PATH}')

## Quick check on a small area (run this before the batch)
Harmonizes a small bbox in memory and runs the daily water balance with a spin-up from field capacity. Plots the harmonized inputs (cropland share, soil water capacity), the status map, and the daily rain, ET₀ and f of a few pixels. Use it to check the crop settings (`CROP`, `KC`, `ZR`, `P_DEPLETION`) and that f behaves sensibly for your region.

In [ ]:
# small AOI inside a cropping area (w, s, e, n); change to any area you know
TEST_BBOX   = (120.90, 15.60, 120.95, 15.65)          # e.g. Nueva Ecija
TEST_PERIOD = RUN_PERIODS[-1].label

qc = runner.quick_check(TEST_BBOX, TEST_PERIOD)
crop = qc['crop_mask'].values == 1
print(f"{qc.sizes['y']} x {qc.sizes['x']} px, {crop.sum()} cropland px | period {qc.attrs['period_start']} .. {qc.attrs['period_end']}")
_i0 = list(pd.DatetimeIndex(qc['time'].values)).index(pd.Timestamp(qc.attrs['period_start']))
print(f'precipitation: {PRECIP_SOURCE.upper()} | days in the period with Late-run (provisional) GPM:',
      int(qc['p_provisional'][_i0:].sum()) if 'p_provisional' in qc else 0,
      '| days with ERA5 precipitation (source not out yet):', int(qc['p_fallback'][_i0:].sum()))
st = qc['status'].values
print(pd.Series(st[crop]).map(bal.STATUS).value_counts())
# sanity check: ET0 in the humid tropics is ~3-6 mm/day; ~0 means the weather data did not come through
_et0 = pd.Series(np.nanmean(qc['ET0'].values[:, crop], axis=1), index=pd.DatetimeIndex(qc['time'].values))
_et = pd.Series(np.nanmean(qc['ET'].values[:, crop], axis=1), index=_et0.index)
display(pd.DataFrame({'ET0 mm/day': _et0.resample('MS').mean(), 'ETc mm/day': _et.resample('MS').mean()}).round(2).T)

In [ ]:
# -1 no data | 1 severe | 2 moderate | 3 adequate | 4 surplus
STATUS_COLORS = {-1: '#d9d9d9', 1: '#c0392b', 2: '#f39c12', 3: '#27ae60', 4: '#2e86c1'}
def plot_status(status, ax, title=''):
    codes = sorted(STATUS_COLORS)
    idx = np.searchsorted(codes, status)
    ax.imshow(idx, cmap=ListedColormap([STATUS_COLORS[c] for c in codes]), vmin=-0.5, vmax=len(codes) - 0.5,
              interpolation='nearest')
    ax.set_title(title); ax.axis('off')

def status_legend(fig):
    fig.legend(handles=[Patch(color=c, label=f'{k} {bal.STATUS[k]}') for k, c in STATUS_COLORS.items()],
               loc='lower center', ncol=5, fontsize=9)

fig, axs = plt.subplots(1, 4, figsize=(19, 4.8))
im = axs[0].imshow(qc['crop_frac'].values, cmap='YlGn', vmin=0, vmax=1); axs[0].set_title('cropland share (WorldCover → 1 ha)')
plt.colorbar(im, ax=axs[0], shrink=0.8)
im = axs[1].imshow(np.where(crop, qc['TAW'].values, np.nan), cmap='Blues'); axs[1].set_title('TAW = FC − WP (mm, root zone)')
plt.colorbar(im, ax=axs[1], shrink=0.8)
im = axs[2].imshow(qc['f_end'].values, cmap='RdYlBu', vmin=0, vmax=1.2); axs[2].set_title('f at period end')
plt.colorbar(im, ax=axs[2], shrink=0.8)
plot_status(qc['status'].values, axs[3], f'status {TEST_PERIOD}')
for a in axs[:3]: a.axis('off')
status_legend(fig); plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

In [ ]:
def plot_pixel(qc, iy, ix, ax):
    t = pd.DatetimeIndex(qc['time'].values)
    p_ = qc.attrs['p']
    P_ = qc['P'].values[:, iy, ix]
    ax.bar(t, P_, color='tab:blue', alpha=0.5, label='P (mm/day)')
    ax.plot(t, qc['ET0'].values[:, iy, ix], color='tab:orange', lw=1, alpha=0.8, label='ET0 (mm/day)')
    ax.plot(t, qc['ET'].values[:, iy, ix], color='tab:red', lw=1.5, label='ETc (mm/day)')
    # cap the axis so ET (a few mm/day) stays readable; tall rain bars are clipped
    wet = P_[P_ > 1]
    ax.set_ylim(0, max(20.0, 1.5 * float(np.nanmax(qc['ET0'].values[:, iy, ix])),
                       float(np.percentile(wet, 75)) if wet.size else 0.0))
    if qc['I'].values[:, iy, ix].any():
        ax.bar(t, qc['I'].values[:, iy, ix], color='tab:cyan', alpha=0.6, label='I (mm/day)')
    ax.set_ylabel('mm/day')
    a2 = ax.twinx()
    a2.plot(t, qc['f'].values[:, iy, ix], color='k', lw=2, label='f')
    a2.axhline(1, color='0.4', lw=0.8, ls=':')
    a2.axhline(1 - p_, color='tab:orange', lw=1, ls='--', label=f'1 − p = {1 - p_:.2f}')
    a2.axhline(qc.attrs['severe_f'], color='tab:red', lw=1, ls='--', label='severe')
    a2.set_ylim(0, 1.4); a2.set_ylabel('f')
    ax.axvspan(pd.Timestamp(qc.attrs['period_start']), pd.Timestamp(qc.attrs['period_end']), color='0.9', zorder=0)
    code_ = int(qc['status'].values[iy, ix])
    ax.set_title(f'pixel ({iy},{ix})  TAW {float(qc["TAW"].values[iy, ix]):.0f} mm  → {bal.STATUS[code_]}', fontsize=9)
    return a2

# a few cropland pixels with different outcomes
rng = np.random.default_rng(0)
picks = []
for code_ in (1, 2, 3, 4):
    cand = np.argwhere(st == code_)
    if len(cand):
        picks.append(cand[rng.integers(len(cand))])
while len(picks) < 4 and crop.any():
    picks.append(np.argwhere(crop)[rng.integers(crop.sum())])
fig, axs = plt.subplots(2, 2, figsize=(15, 7.5))
for ax, (iy, ix) in zip(axs.ravel(), picks):
    a2 = plot_pixel(qc, iy, ix, ax)
h1, l1 = axs[0, 0].get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc='lower center', ncol=7, fontsize=8)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

In [ ]:
# area-average daily balance of the test area (cropland pixels)
mean = lambda v: np.nanmean(qc[v].values[:, crop], axis=1)
daily = pd.DataFrame({v: mean(v) for v in ('P', 'ET0', 'ET', 'R', 'D', 'I', 'S', 'f')}, index=pd.DatetimeIndex(qc['time'].values))
in_period = daily.loc[qc.attrs['period_start']:qc.attrs['period_end']]
print('period totals (mm):', in_period[['P', 'ET0', 'ET', 'R', 'D', 'I']].sum().round(1).to_dict())
print('ΔS = %.1f mm, f: %.2f -> %.2f' % (in_period['S'].iloc[-1] - daily['S'].shift(1).loc[in_period.index[0]],
                                          daily['f'].shift(1).loc[in_period.index[0]], in_period['f'].iloc[-1]))
daily[['f']].plot(figsize=(12, 3), title='mean f (cropland)'); plt.show()

## Outputs
Products are written under the output root: `DRIVE_OUTPUT_ROOT` for `gdrive` / `both`, or `LOCAL_OUTPUT_DIR` for `local`. With `gcs` / `both` they're copied to `gs://GCS_BUCKET/GCS_PREFIX/…`. The folder layout is the same everywhere: `{AREA}/{PERIOD}/wa_{AREA}_{PERIOD}_*`.

In [ ]:
print('destination:', cfg.output_root(), f'+ gs://{GCS_BUCKET}/{GCS_PREFIX}' if NEED_GCS else '')
# plan: tiles per area and how many still need a spin-up (no saved soil state yet)
if AREA_LEVEL == 'bbox':
    display(pd.DataFrame(runner.plan(RUN_PERIODS)['periods']))
else:
    areas = [n for n, _ in runner.select_areas(AREA_LEVEL, AREA_NAMES) if not any(x in n for x in EXCLUDE)]
    plans = [{'area': n, **row} for n, g in runner.select_areas(AREA_LEVEL, AREA_NAMES) if n in areas
             for row in runner.area_runner(n, g).plan(RUN_PERIODS)['periods']]
    display(pd.DataFrame(plans))

## Periodic run (national / regional / provincial / municipal)

In [ ]:
# areas are processed one by one, tile by tile (harmonize -> water balance), periods in date order
if AREA_LEVEL == 'bbox':
    summaries = runner.run(RUN_PERIODS)
    results = {'success': [{'area': s['aoi'], **s} for s in summaries], 'failed': [], 'mosaics': []}
else:
    only = [n for n, _ in runner.select_areas(AREA_LEVEL, AREA_NAMES) if not any(x in n for x in EXCLUDE)]
    results = runner.run_areas(AREA_LEVEL, AREA_NAMES, RUN_PERIODS, only=only)
# retry only the failed areas, e.g.:
# results = runner.run_areas(AREA_LEVEL, AREA_NAMES, RUN_PERIODS, only=[f['area'] for f in results['failed']])

In [ ]:
rows = ([{'area': s['area'], 'period': s['period'], 'status': 'success',
          'tiles_with_cropland': s['tiles_with_cropland'], 'spun_up': s['tiles_spun_up'],
          'failed_tiles': len(s['failed_tiles']), 'provisional': s['provisional'],
          'output': s.get('uris', [s['outputs']['status']])[0].rsplit('/', 1)[0]} for s in results['success']]
        + [{'area': f['area'], 'status': 'failed', 'error': f['error']} for f in results['failed']])
res = pd.DataFrame(rows)
print(res['status'].value_counts())
display(res)

In [ ]:
# hectares per status class
for s in results['success']:
    print(f"{s['area']} — {s['period']}" + (f"  (provisional: {s['precip_provisional_days']} Late-run GPM day(s), "
                                          f"{s['precip_from_era5_days']} ERA5 day(s))" if s['provisional'] else ''))
    display(pd.DataFrame(s['summary']).set_index('class')[['hectares', 'cropland_hectares', 'share_pct']])

In [ ]:
# mosaic: all areas of the selection -> one COG per period (+ hectares CSV) in <output root>/<MOSAIC_LABEL>/
# run_areas already did this when MOSAIC_AFTER is on; run this to rebuild it, e.g. after retrying failures
mz = results.get('mosaics') or []
# mz = runner.mosaic(MOSAIC_LABEL or 'PHILIPPINES', [n for n, _ in runner.select_areas(AREA_LEVEL, AREA_NAMES)], RUN_PERIODS)
for m in mz:
    if 'error' in m:
        print(m['period'], m['error']); continue
    print(f"{m['period']}: {m['areas']} areas -> {m['status']}")
    display(pd.DataFrame(m['summary']).set_index('class')[['hectares', 'share_pct']])
    if m['missing']:
        print('areas without a map yet (not in the mosaic):', m['missing'])

## Look at the maps
Reads the status and balance GeoTIFFs back. They're on Drive or local disk, or in the GCS staging folder for `both`.

In [ ]:
import rasterio

def show_area(stem):
    with rasterio.open(stem + '_status.tif') as src:
        status = src.read(1)
    with rasterio.open(stem + '_balance.tif') as src:
        names = list(src.descriptions)
        band = lambda n: src.read(names.index(n) + 1)
        f_end, P, ET, dS = band('f_end'), band('P'), band('ET'), band('dS')
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    plot_status(status, axs[0], os.path.basename(stem))
    for ax, (arr, title, cmap, lim) in zip(axs[1:], [(f_end, 'f at period end', 'RdYlBu', (0, 1.2)),
                                                     (P - ET, 'P − ET (mm)', 'BrBG', (-60, 60)),
                                                     (dS, 'ΔS (mm)', 'BrBG', (-60, 60))]):
        im = ax.imshow(arr, cmap=cmap, vmin=lim[0], vmax=lim[1]); ax.set_title(title); ax.axis('off')
        plt.colorbar(im, ax=ax, shrink=0.7)
    status_legend(fig); plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

if OUTPUT_TARGET != 'gcs':
    for s in results['success'][:6]:
        show_area(s['outputs']['status'].replace('_status.tif', ''))
    for m in mz:
        if 'status' in m:
            show_area(m['status'].replace('_status.tif', ''))

## Every 14 days
Each scheduled run makes the newest finished period: leave `PERIODS = None` and rerun the notebook.
* The saved soil state in `WORK_DIR` lets the run start from the previous period's end, so it downloads only 14 days.
* Keep `WORK_DIR` on Drive so the state survives between Colab sessions.
* Run periods in date order. To backfill, list the labels, e.g. `PERIODS = ['20261008', '20261022']`.
* With GPM, a fresh period is available a day or two after it ends. It's made from IMERG's **Late run** and marked `provisional` (`precip_provisional_days` in `_run.json`). IMERG's **Final run** replaces the Late run a few months later. Rerun those periods then, **and every later one, in order** (e.g. `PERIODS = ['20261008', '20261022']`). Provisional tiles are always fetched again rather than taken from the cache.
* Days with no precipitation at all yet use ERA5-Land precipitation (`precip_from_era5_days`), also provisional.

The same runs from the command line:
```
python -m waterbalance run --config config.json                       # last complete period
python -m waterbalance run --config config.json --periods 20261022
```